# 05 Test results
Latest full run per model, split by training audio (raw vs trimmed). EER recomputed from each run's score files.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from audio_df.helpers.metrics import compute_eer

MODELS = ["a1", "a4", "a2", "d1", "d2", "d3"]
ART = Path("../artifacts")

def eer(df):
    return 100 * compute_eer(df.score, df.label, higher_is_spoof=False)[0]

def train_audio(artifact):
    p = Path(artifact) / "preprocessing.json"
    return json.loads(p.read_text())["train"] if p.exists() else "raw"

runs = []
for d in sorted(Path("../runs").glob("E-*")):
    cfg = d / "config.yaml"
    c = yaml.safe_load(cfg.read_text()) if cfg.exists() else {}
    if c.get("architecture") in MODELS and (d / "scores/itw.csv").exists():
        runs.append({"model": c["architecture"], "train": train_audio(c["model_artifact"]), "run": d})
runs = pd.DataFrame(runs).groupby(["model", "train"]).last().reset_index()
runs.assign(run=runs.run.map(lambda p: p.name))

## EER % per test set

In [ ]:
scores = {(r.model, r.train): {p.stem: pd.read_csv(p) for p in sorted(r.run.glob("scores/*.csv"))}
          for r in runs.itertuples()}
table = pd.DataFrame({k: {t: eer(s) for t, s in v.items()} for k, v in scores.items()}).T
table.index.names = ["model", "train"]
table.loc[[(m, t) for m in MODELS for t in ["raw", "trimmed"] if (m, t) in table.index]].round(2)

## Per attack, eval (EER %)

In [ ]:
def per_attack(df):
    bona = df[df.label == "bonafide"]
    return {a: eer(pd.concat([bona, g])) for a, g in df[df.label == "spoof"].groupby("attack_id")}

pd.DataFrame({k: per_attack(v["eval"]) for k, v in scores.items()}).T.round(1)

## Score distributions
Higher = bona fide. A model that collapses puts both classes on the same side.

In [ ]:
sets = ["dev", "eval", "itw"]
fig, axes = plt.subplots(len(scores), len(sets), figsize=(12, 1.8 * len(scores)), squeeze=False)
for i, (k, v) in enumerate(scores.items()):
    for j, t in enumerate(sets):
        ax = axes[i, j]
        for lab, color in [("bonafide", "tab:green"), ("spoof", "tab:orange")]:
            ax.hist(v[t].score[v[t].label == lab], bins=60, density=True, histtype="step", color=color, label=lab)
        ax.set_title(f"{k[0]} ({k[1]} train) - {t}", fontsize=9)
        ax.set_yticks([])
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()